# Example 02: Weak Signal Detection

Feed time-series data to the WeakSignalDetector and catch precursor patterns
before events become obvious in the news.

In [ ]:
import pandas as pd
import numpy as np
from mimic_signal.weak_signals import WeakSignalDetector, ALL_PATTERNS

# Show all available patterns
for p in ALL_PATTERNS:
    print(f"{p.name}")
    print(f"  Precedes: {p.precedes}")
    print(f"  Lead time: {p.lead_time_days[0]}-{p.lead_time_days[1]} days")
    print(f"  Historical precision: {p.historical_precision:.0%}")
    print()

In [ ]:
# Set up detector watching all patterns
detector = WeakSignalDetector()
detector.watch_all()

fired_signals = []

@detector.on_weak_signal("bdi_decline_precedes_shipping_disruption")
def on_bdi(ws):
    fired_signals.append(ws)
    print(f"\nWEAK SIGNAL DETECTED: {ws.title}")
    print(f"  Precedes:       {ws.precedes}")
    print(f"  Lead time:      {ws.lead_time_estimate} days (range: {ws.lead_time_min_days}-{ws.lead_time_max_days})")
    print(f"  Confidence:     {ws.confidence:.0%}")
    print(f"  Signal value:   {ws.signal_value:.2f}")

In [ ]:
# Simulate a BDI decline (2000 → 1500 over 11 trading days)
bdi_values = np.linspace(2000, 1480, 11).tolist()
print("Feeding BDI values:", [f"{v:.0f}" for v in bdi_values])

ws_list = detector.update_series("bdi_daily", bdi_values)
print(f"\nWeak signals triggered: {len(ws_list)}")

In [ ]:
# Simulate vessel queue growth at a port
@detector.on_weak_signal("vessel_queue_growth_precedes_port_disruption")
def on_vessel_queue(ws):
    print(f"PORT SIGNAL: {ws.title}")
    print(f"  Lead time: {ws.lead_time_min_days}-{ws.lead_time_max_days} days")

# Normal: ~20 vessels, then queue grows to 40
vessel_counts = [20, 20, 21, 22, 23, 24, 36]
detector.update_series("ais_vessel_queue", vessel_counts)

In [ ]:
# Review fired weak signals
print(f"Total weak signals fired: {len(fired_signals)}")
for ws in fired_signals:
    print(f"  {ws}")